# Machine-Translation Agents with `smolagents`

*DGT Summer School 2026 — building a translator's assistant that orchestrates CAT tools.*

A **machine-translation (MT) agent** is a language model that can *decide when to
reach for a tool* — a term base, a translation memory (TM), an MT engine, a
quality-assurance (QA) check — the same way a human translator works inside a CAT
environment. Instead of translating in one shot, the agent reasons step by step,
calls the right aid at the right moment, and checks its own work.

This notebook builds that idea up from a single tool to a full multi-agent
translation pipeline, using `smolagents` and the OpenAI API. It is a translation
counterpart to `Introduction_to_agents.ipynb` — the agent *concepts* are the same;
the examples are all about translation.

## 1  What is a translation agent?

Three building blocks, from smallest to largest:

| Concept | In translation terms |
|---|---|
| **Tool** | One concrete action: look up a term, query the TM, translate a segment, run a QA check. |
| **Agent** | A model that plans, calls tools in sequence, reads their output, and produces a final translation. |
| **Multi-agent system** | Several specialised agents (terminology, translation, review) coordinated by an orchestrator — a small translation workflow. |

We will meet each in turn, ending with a pipeline that routes a source segment
through terminology preparation, translation, and review.

In [ ]:
# !pip install -q -U smolagents openai ddgs python-dotenv
# The uv environment already provides these — this cell is here only for reference.

## 2  Setup

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

# Load OPENAI_API_KEY from a local .env file (which must NOT be committed).
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - add it to your .env file."

from openai import OpenAI
from smolagents import (
    CodeAgent,
    ToolCallingAgent,
    OpenAIServerModel,
    PythonInterpreterTool,
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    tool,
)
from smolagents.memory import ActionStep, FinalAnswerStep, SystemPromptStep, TaskStep

## 3  Connecting the model

`OpenAIServerModel` wraps the OpenAI Chat Completions API in the `smolagents`
`Model` interface, so the agents can talk to `gpt-4o-mini`. We *also* create a raw
`OpenAI` client — some of our tools call the translation service directly, which is
a realistic "tool wraps an external API" pattern.

In [ ]:
# Model id and endpoint are read from .env (OPENAI_MODEL, OPENAI_BASE_URL),
# so the same code works against OpenAI or the course's custom endpoint.
MODEL_ID = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")

model = OpenAIServerModel(
    model_id=MODEL_ID,
    api_base=BASE_URL,
    api_key=os.environ["OPENAI_API_KEY"],
)

# A raw OpenAI client for the tools that call the MT / LLM service directly.
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=BASE_URL)

print(f"Model ready: {MODEL_ID}  (endpoint: {BASE_URL})")

## 4  A first agent, with no tools

Given nothing but the model, an agent can already detect a language and translate
from its own knowledge. Below it receives a French sentence and must translate it
into English, flagging any ambiguity. Watch the reasoning trace: it plans, acts,
and returns a final answer.

In [7]:
agent = ToolCallingAgent(
    tools=[],
    model=model,
    max_steps=4,
)

result = agent.run(
    "Detect the language of the sentence below, then translate it into English. "
    "Point out any word whose translation is ambiguous and why.\n\n"
    "« La souris a mangé le fromage sur le bureau. »"
)
print("\nFinal answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Detect the language of the sentence below, then translate it into English. Point out any word whose translation │
│ is ambiguous and why.                                                                                           │
│                                                                                                                 │
│ « La souris a mangé le fromage sur le bureau. »                                                                 │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The detected language is French.'}                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The detected language is French.

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The translation of the sentence is: "The mouse ate the │
│ cheese on the desk."'}                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The language of the sentence is French. The            │
│ translation into English is "The mouse ate the cheese on the desk." The word \'bureau\' is ambiguous because it │
│ can mean \'desk\' or \'office\' in English, depending on the context.'}                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The language of the sentence is French. The translation into English is "The mouse ate the cheese on 
the desk." The word 'bureau' is ambiguous because it can mean 'desk' or 'office' in English, depending on the 
context.

Final answer: The language of the sentence is French. The translation into English is "The mouse ate the cheese on 
the desk." The word 'bureau' is ambiguous because it can mean 'desk' or 'office' in English, depending on the 
context.

[Step 2: Duration 1.37 seconds| Input tokens: 1,803 | Output tokens: 143]


Final answer: The language of the sentence is French. The translation into English is "The mouse ate the cheese on the desk." The word 'bureau' is ambiguous because it can mean 'desk' or 'office' in English, depending on the context.


The bare agent translates fine, but it has **no term base, no translation
memory, and no QA** — so it cannot guarantee approved terminology, reuse previous
work, or check numbers and length. In a professional workflow those guarantees are
the whole point. That is what tools give us.

## 5  Building the translation toolbox

We now write five custom tools with the `@tool` decorator. The decorator turns an
annotated Python function into something the agent can discover and call — the
function name, docstring, and type hints become the tool's interface.

- `machine_translate` — calls the MT engine (a live LLM call).
- `detect_language` — identifies the source language.
- `glossary_lookup` — approved translations from an IATE-style term base.
- `translation_memory_lookup` — fuzzy match against previously translated segments.
- `qa_check` — automatic quality checks on a translation.

In [ ]:
@tool
def machine_translate(text: str, source_lang: str, target_lang: str) -> str:
    """Translate text from one language to another using an MT engine.

    This tool wraps a live call to an external translation service (an LLM).

    Args:
        text: The source text to translate.
        source_lang: ISO 639-1 code of the source language (e.g. "en", "fr", "de").
        target_lang: ISO 639-1 code of the target language (e.g. "en", "fr", "de").
    """
    response = client.chat.completions.create(
        model=MODEL_ID,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional translator. Translate the user's text "
                    f"from {source_lang} to {target_lang}. Return only the translation, "
                    "with no explanations and no surrounding quotation marks."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    return response.choices[0].message.content.strip()


# Inspect what the model will see when deciding whether to call this tool
print("Tool name  :", machine_translate.name)
print("Description:", machine_translate.description)
print("Inputs     :", machine_translate.inputs)
print("Output type:", machine_translate.output_type)

# Call it directly to confirm it works
print("\nDirect call:", machine_translate("Good morning, everyone.", "en", "de"))

In [ ]:
@tool
def detect_language(text: str) -> str:
    """Detect the language of a piece of text.

    Returns a two-letter ISO 639-1 code such as "en", "fr", "de" or "es".

    Args:
        text: The text whose language should be identified.
    """
    response = client.chat.completions.create(
        model=MODEL_ID,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Identify the language of the user's text. Answer with a single "
                    "ISO 639-1 code (e.g. en, fr, de, es) and nothing else."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    return response.choices[0].message.content.strip().lower()[:2]


for sample in ["Buenos días a todos.", "Guten Morgen zusammen.", "Bonjour à tous."]:
    print(f"{detect_language(sample)}  <-  {sample}")

### 5.1  Deterministic aids: term base and translation memory

Not every tool needs the model. A **term base** (glossary) and a **translation
memory** are just data the tool looks things up in — here backed by small
in-notebook dictionaries. In production these would be IATE and Euramis; the
interface the agent sees is identical.

In [ ]:
# A tiny IATE-style multilingual term base: approved translations for domain terms.
GLOSSARY = {
    "machine translation": {"fr": "traduction automatique", "de": "maschinelle Übersetzung", "es": "traducción automática"},
    "translation memory":  {"fr": "mémoire de traduction", "de": "Translation Memory", "es": "memoria de traducción"},
    "member state":        {"fr": "État membre", "de": "Mitgliedstaat", "es": "Estado miembro"},
    "regulation":          {"fr": "règlement", "de": "Verordnung", "es": "reglamento"},
    "directive":           {"fr": "directive", "de": "Richtlinie", "es": "directiva"},
}


@tool
def glossary_lookup(term: str, target_lang: str) -> dict:
    """Look up the approved translation of a term in the term base (glossary).

    Args:
        term: The source term to look up (English, case-insensitive).
        target_lang: ISO 639-1 code of the target language (e.g. "fr", "de", "es").
    """
    entry = GLOSSARY.get(term.strip().lower())
    if not entry or target_lang not in entry:
        return {"term": term, "found": False, "translation": None}
    return {"term": term, "found": True, "translation": entry[target_lang]}


print(glossary_lookup("Member State", "fr"))
print(glossary_lookup("blockchain", "fr"))   # not in the term base

In [ ]:
from difflib import SequenceMatcher

# A tiny translation memory: previously translated segments (source -> per-language target).
TRANSLATION_MEMORY = {
    "The Commission shall adopt implementing acts.": {
        "fr": "La Commission adopte des actes d'exécution.",
        "de": "Die Kommission erlässt Durchführungsrechtsakte.",
    },
    "This Regulation shall be binding in its entirety.": {
        "fr": "Le présent règlement est obligatoire dans tous ses éléments.",
        "de": "Diese Verordnung ist in allen ihren Teilen verbindlich.",
    },
    "Member States shall bring into force the necessary provisions.": {
        "fr": "Les États membres mettent en vigueur les dispositions nécessaires.",
        "de": "Die Mitgliedstaaten setzen die erforderlichen Vorschriften in Kraft.",
    },
}


@tool
def translation_memory_lookup(segment: str, target_lang: str) -> dict:
    """Find the closest previously translated segment in the translation memory (fuzzy match).

    Returns the best matching source segment, a similarity score between 0 and 1,
    and its stored translation. A score of 1.0 is an exact match; translators
    typically reuse matches above about 0.75.

    Args:
        segment: The source segment to look up.
        target_lang: ISO 639-1 code of the target language (e.g. "fr", "de").
    """
    best_source, best_score = None, 0.0
    for source in TRANSLATION_MEMORY:
        score = SequenceMatcher(None, segment.lower(), source.lower()).ratio()
        if score > best_score:
            best_source, best_score = source, score
    translation = TRANSLATION_MEMORY.get(best_source, {}).get(target_lang)
    return {"best_match": best_source, "score": round(best_score, 2), "translation": translation}


print(translation_memory_lookup("The Commission shall adopt implementing acts.", "fr"))
print(translation_memory_lookup("Member States shall bring into force the provisions.", "de"))

In [ ]:
import re


@tool
def qa_check(source: str, translation: str) -> dict:
    """Run automatic quality checks comparing a translation against its source.

    Checks performed:
    - non_empty: the translation is not empty
    - not_echoed: the translation differs from the source (i.e. it was translated)
    - length_ratio_ok: target/source character ratio is within a sane 0.5-2.0 band
    - numbers_preserved: every number in the source also appears in the translation

    Returns a dict with "passed" (bool) and a list of "issues".

    Args:
        source: The original source text.
        translation: The proposed translation to check.
    """
    issues = []
    if not translation.strip():
        issues.append("translation is empty")
    if translation.strip().lower() == source.strip().lower():
        issues.append("translation is identical to the source (untranslated)")
    ratio = len(translation) / max(1, len(source))
    if not 0.5 <= ratio <= 2.0:
        issues.append(f"suspicious length ratio ({ratio:.2f})")
    src_nums, tgt_nums = set(re.findall(r"\d+", source)), set(re.findall(r"\d+", translation))
    if src_nums - tgt_nums:
        issues.append(f"numbers missing in translation: {sorted(src_nums - tgt_nums)}")
    return {"passed": not issues, "issues": issues}


print(qa_check("The deadline is 15 May 2026.", "Le délai est fixé au 15 mai 2026."))
print(qa_check("The deadline is 15 May 2026.", "The deadline is 15 May 2026."))

## 6  Give the agent a goal, not a script

This is the heart of what makes an agent an *agent*. A chain runs a fixed sequence
of steps that **we** decide. An agent gets only a **goal** and a **toolbox**, and
works out for itself which tools are worth calling, in what order, and how to check
its own answer.

To make the difference visible we hand it one dense, technical paragraph — the kind
where approved terminology, a reusable segment, numbers, and a slippery phrase like
"high-risk" all matter at once — and simply ask for a publication-quality
translation. No numbered recipe. Watch the trace: the plan is the agent's, not
ours.

In [ ]:
translation_agent = CodeAgent(
    tools=[
        detect_language,
        glossary_lookup,
        translation_memory_lookup,
        machine_translate,
        qa_check,
        FinalAnswerTool(),
    ],
    model=model,
    code_block_tags="markdown",
    max_steps=12,
)

paragraph = (
    "Under the new Regulation, each Member State shall designate a competent authority "
    "to audit high-risk machine translation systems before 15 May 2026. The Commission "
    "may adopt implementing acts specifying the conformity-assessment procedure, "
    "including thresholds for post-editing effort and translation-memory reuse rates "
    "above 75%."
)

result = translation_agent.run(
    "Produce a publication-quality French translation of the paragraph below for an "
    "official EU document.\n\n"
    "You have tools for language detection, terminology lookup, translation-memory "
    "search, machine translation, and quality assurance. Decide for yourself which of "
    "them are worth using and in what order — you do not have to use all of them. Make "
    "the terminology consistent with the approved term base, reuse a translation-memory "
    "match if one is close enough, and verify your own translation before returning it. "
    "Briefly explain which tools you used and why.\n\n"
    f"Paragraph: {paragraph}"
)
print("\nFinal answer:", result)

In [ ]:
# A compact trace of what the agent did at each step
action_steps = [s for s in translation_agent.memory.steps if isinstance(s, ActionStep)]
print(f"Number of action steps: {len(action_steps)}\n")

for step in action_steps:
    print(f"--- Step {step.step_number} ---")
    print("Code :", (step.code_action or "").strip()[:120])
    print("Obs  :", str(step.observations or "")[:100])
    print()

Because we specified a *goal* and not a *script*, the trace above is the
agent's own plan. It decided which terms to send to the term base, judged whether
the translation-memory match was close enough to reuse, and chose how to verify —
typically a `qa_check` confirming that the date and the "75%" survived. Run the cell
again and the plan may come out slightly different; that adaptivity is exactly what
separates an agent from a fixed pipeline. If you *want* determinism, you constrain
it in the prompt — but then you are writing a chain, not delegating to an agent.

## 7  Looking up official terminology on the web

Some names are not in your term base — the *officially* correct rendering of an
institution or programme in the target language. Rather than telling the agent to
search, we just tell it the quality bar ("use the official designation, verify if
unsure") and give it `DuckDuckGoSearchTool`. It decides for itself whether it needs
to look something up.

In [ ]:
search = DuckDuckGoSearchTool()

# Call the tool directly to see what the agent will read
print(search("European Data Protection Supervisor official French name")[:600])

In [ ]:
agent_terminology = CodeAgent(
    tools=[DuckDuckGoSearchTool(), machine_translate, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=6,
)

result = agent_terminology.run(
    "Translate the sentence below into French for an official EU document. Any "
    "institution name must use its correct official French designation — if you are "
    "not certain of it, verify it rather than guessing.\n\n"
    "Sentence: The European Data Protection Supervisor published new guidelines."
)
print("\nFinal answer:", result)

## 8  How the agent chooses tools: `CodeAgent` vs `ToolCallingAgent`

`smolagents` offers two agent styles. A **`CodeAgent`** writes Python code that
calls tools as functions; a **`ToolCallingAgent`** emits structured JSON tool
calls (OpenAI-native function calling). Both solve the same translation task —
compare how they express the *same* tool call.

In [ ]:
TASK = (
    "Translate 'The meeting is postponed.' into German. "
    "Use the machine_translate tool with source_lang 'en' and target_lang 'de'."
)

# --- CodeAgent: writes code that calls the tool ---
code_agent = CodeAgent(
    tools=[machine_translate, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=4,
)
print("CodeAgent answer:", code_agent.run(TASK))

# --- ToolCallingAgent: emits structured JSON tool calls ---
tc_agent = ToolCallingAgent(
    tools=[machine_translate, FinalAnswerTool()],
    model=model,
    max_steps=4,
)
print("ToolCallingAgent answer:", tc_agent.run(TASK))

In [ ]:
# Compare how each agent expresses its first action.
# A CodeAgent records a code block in `model_output`; a ToolCallingAgent records a
# structured call in `tool_calls`.
def first_action(agent):
    for step in agent.memory.steps:
        if isinstance(step, ActionStep):
            # CodeAgent records a code block here; for a ToolCallingAgent it is empty.
            if step.model_output and step.model_output.strip():
                return step.model_output.strip()
            # ToolCallingAgent records its structured call here instead.
            if step.tool_calls:
                return "\n".join(f"{c.name}(arguments={c.arguments})" for c in step.tool_calls)
    return "(no action step found)"

print("=== CodeAgent - writes a Python code block ===")
print(first_action(code_agent)[:400])

print("\n=== ToolCallingAgent - emits a structured tool call ===")
print(first_action(tc_agent)[:400])

The `CodeAgent` output contains a Python code block calling
`machine_translate(...)`; the `ToolCallingAgent` output is a JSON tool call. Same
intent, two encodings. For most translation workflows either works — `CodeAgent`
shines when a step needs light glue logic (looping over segments, combining tool
results), which is common in batch translation.

We can also inspect an agent's **memory** to debug what happened — every step is
recorded.

In [ ]:
print(f"Total steps in memory: {len(translation_agent.memory.steps)}\n")

for i, step in enumerate(translation_agent.memory.steps):
    step_type = type(step).__name__
    print(f"[{i}] {step_type}")
    if isinstance(step, SystemPromptStep):
        print("    ", step.system_prompt.splitlines()[0][:80])
    elif isinstance(step, TaskStep):
        print("    Task:", step.task[:80])
    elif isinstance(step, ActionStep):
        print("    Code :", (step.code_action or "").strip()[:70])
        print("    Obs  :", str(step.observations or "")[:60])
    elif isinstance(step, FinalAnswerStep):
        print("    Output:", str(step.output)[:80])

## 9  A multi-agent translation team

Real translation work is divided by role: a terminologist prepares references, a
translator drafts, a reviewer checks. We mirror that with three specialised agents,
each owning a slice of the toolbox, coordinated by an **orchestrator**. To the
orchestrator, each sub-agent behaves like a callable tool (via `managed_agents`).

Same principle as before: we tell the orchestrator the *goal* and who is available,
not a call sequence — it decides how to delegate. We give it the *same* hard
paragraph from section 6, so you can compare one agent doing everything against a
team dividing the labour.

In [ ]:
# Sub-agent 1: terminology specialist — prepares a term/TM brief
terminology_agent = CodeAgent(
    tools=[glossary_lookup, translation_memory_lookup, DuckDuckGoSearchTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    name="terminology_agent",
    description=(
        "Prepares terminology for a source segment. Give it an English segment and a "
        "target language; it returns approved term-base translations and the closest "
        "translation-memory match."
    ),
    max_steps=6,
    verbosity_level=0,
)

# Sub-agent 2: translator — produces the draft translation
translator_agent = CodeAgent(
    tools=[detect_language, machine_translate, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    name="translator_agent",
    description=(
        "Translates a segment. Give it a source segment, a target language, and any "
        "terminology guidance; it returns the translation."
    ),
    max_steps=6,
    verbosity_level=0,
)

# Sub-agent 3: reviewer — QA and back-translation
reviewer_agent = CodeAgent(
    tools=[qa_check, machine_translate, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    name="reviewer_agent",
    description=(
        "Reviews a translation. Give it the source and the proposed translation; it "
        "runs QA checks and can back-translate to verify meaning, then reports issues."
    ),
    max_steps=6,
    verbosity_level=0,
)

# Orchestrator: routes the segment through terminology -> translation -> review
orchestrator = CodeAgent(
    tools=[FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    managed_agents=[terminology_agent, translator_agent, reviewer_agent],
    max_steps=10,
)

result = orchestrator.run(
    "Produce a verified, publication-quality French translation of the paragraph below "
    "for an official EU document. You coordinate three specialists: a terminology "
    "agent, a translator agent, and a reviewer agent. Delegate to them however you "
    "judge best so that the terminology is correct and the final translation has been "
    "checked. Return the final French translation and the reviewer's verdict.\n\n"
    f"Paragraph: {paragraph}"
)
print("\nFinal answer:", result)

The orchestrator treated each sub-agent as a tool and decided how to sequence
them, combining their results — terminology feeding the translator, whose draft
feeds the reviewer. This is the same division of labour as a human translation team,
and it scales: swap in a stronger MT engine, a real IATE connector, or a second
reviewer without touching the others.

## Summary

- The defining idea: an agent is given a **goal, not a script**. We describe the
  outcome and the available tools; the agent decides which to call, in what order,
  and how to verify. Spell out the steps yourself and you have built a fixed chain,
  not an agent.
- A **tool** is one translation action; the `@tool` decorator exposes an annotated
  Python function to the agent.
- Tools can wrap **live services** (`machine_translate`, `detect_language`) or
  **local data** (`glossary_lookup`, `translation_memory_lookup`, `qa_check`).
- A **`CodeAgent`** composes tools by writing code; a **`ToolCallingAgent`** emits
  JSON tool calls. Both do **multi-step** tool use and keep an inspectable
  **memory**.
- Web search supplies **official terminology** the term base lacks.
- A **multi-agent pipeline** (terminology → translation → review) mirrors a real
  CAT workflow and keeps each role independently improvable.

Next steps: connect the tools to real back-ends (IATE, Euramis, a production MT
engine), batch over a document's segments, and add human-in-the-loop post-editing.

## References

- [smolagents documentation](https://huggingface.co/docs/smolagents)
- [OpenAI API reference](https://platform.openai.com/docs/api-reference)
- [IATE — the EU's terminology database](https://iate.europa.eu/)